# 🎙️ eidcloud-audio-inspector: Quickstart Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shadialhasan/eidcloud-audio-inspector/blob/main/notebooks/quickstart.ipynb)

> **Audio Forensics & Broadcast Compliance Analyzer** measuring Integrated LUFS, True Peak (dBTP), Loudness Range (LRA), RMS energy, Signal-to-Noise Ratio (SNR), and silence ratios via FFmpeg filter graphs and pure PHP 8.2+ mathematical fallbacks.

This interactive notebook demonstrates:
1. Environment verification (PHP 8.2+ and FFmpeg).
2. Generating calibrated reference audio signals and broadcast tones.
3. Running `eidcloud-audio` CLI inspection and compliance checking against EBU R128 and YouTube standards.

In [ ]:
# Step 1: Install system packages & clone repository if on Google Colab
import os

if not os.path.exists('bin/eidcloud-audio'):
    !git clone https://github.com/shadialhasan/eidcloud-audio-inspector.git
    %cd eidcloud-audio-inspector

!apt-get update -qq && apt-get install -qq -y php-cli ffmpeg > /dev/null
!php -v
!ffmpeg -version | head -n 1

# Step 2: Generate dynamic broadcast reference tone and inspect via CLI
!php bin/eidcloud-audio demo

In [ ]:
# Step 3: Run machine-readable JSON inspection against YouTube & EBU R128
import json
import subprocess
import pandas as pd

# Generate calibrated -14 LUFS streaming tone in pure PHP
!php -r "require 'src/Synthesizer/WavSynthesizer.php'; \EidCloud\AudioInspector\Synthesizer\WavSynthesizer::generateSineWave('streaming_master.wav', 1000.0, 3.0, -14.0);"

# Inspect against YouTube streaming standard with --json flag
cmd = ['php', 'bin/eidcloud-audio', 'inspect', 'streaming_master.wav', '--standard=youtube', '--json']
res = subprocess.run(cmd, capture_output=True, text=True)

data = json.loads(res.stdout)
metrics = data['metrics']['acoustic_metrics']
comp = data['compliance']

print(f"=== Compliance Result for {comp['standard']['name']} ===")
print(f"Status:           {comp['status']}")
print(f"Measured Loudness: {metrics['integrated_lufs']} LUFS (Target: {comp['summary']['target_lufs']} LUFS)")
print(f"True Peak:        {metrics['true_peak_dbtp']} dBTP (Max: {comp['summary']['max_true_peak_dbtp']} dBTP)")
print(f"Dynamic SNR:      {metrics['snr_db']} dB")
print(f"Clipped Samples:  {metrics['clipped_samples']}")

# Display Checks Matrix as DataFrame
df_checks = pd.DataFrame(comp['checks'])
display(df_checks[['name', 'passed', 'target', 'actual', 'note']])